In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from google.colab import drive
import pandas as pd

drive.mount('/content/drive')
current_dir = '/content/drive/MyDrive/Colab Notebooks/Tesi'
%cd $current_dir

In [ ]:
file = "per_martina_new_finally_updated.xlsx"
data = pd.read_excel(file, sheet_name=None)
print(data.keys())
data_crf = pd.read_excel(file, sheet_name="CRF")
data_dose = pd.read_excel(file, sheet_name="dosimetria")
data_metab = pd.read_excel(file, sheet_name="features metaboliche")
data_vol = pd.read_excel(file, sheet_name="HSs_volumi (cm^3)")
print(data_crf.shape)
print(data_dose.shape)
print(data_metab.shape)
print(data_vol.shape)

In [ ]:
#per plots
#plt.hist(df['colonna'])
#plt.scatter(range(len(df)), df['colonna'])
#sns.countplot(data=df, x='colonna_categorica')
#plt.show()

# **Step 1: fix data structure**
**data**: all sheets


**data1, data2, data3**: CRF, DOSE, METABOLIC sheet with fixed cols


**data_crf**: CRF sheet with fixed cols and aligned ID


**data_dose**: DOSE sheet with fixed cols and aligned ID


**data_metab**: METABOLIC sheet with fixed cols and aligned ID


**data123**: merged(data_crf, data_dose, data_metab)


### **Removing unamed columns, duplicates + adding "cycles_chemo_sum"**

In [ ]:
print(data_crf.shape)
print(data_dose.shape)
print(data_vol.shape)
print(data_metab.shape)
print(data_crf.columns)
print(data_dose.columns)
print(data_vol.columns)
print(data_metab.columns)

In [ ]:
print(data_crf.shape)
print(data_dose.shape)
print(data_vol.shape)
print(data_metab.shape)
print(data_crf.columns)
print(data_dose.columns)
print(data_vol.columns)
print(data_metab.columns)

#drop duplicated ID
data_metab = data_metab.drop(columns=["heart_agatston"])
data_metab = data_metab.drop(columns=["heart_volume_mm3"])
data_metab = data_metab.drop(columns=["aorta_agatston"])
data_metab = data_metab.drop(columns=["aorta_volume_mm3"])


#renaming
data_crf = data_crf.rename(columns={"Unnamed: 57": "Note"})
data_crf = data_crf.rename(columns={"Immunotherapy": "IMMUNO"})
data_crf = data_crf.rename(columns={"cycle_immuno": "cycles_immuno"})
data_metab = data_metab.rename(columns={"aorta_agatston.1": "aorta_agatston"})
data_metab = data_metab.rename(columns={"aorta_volume_mm3.1": "aorta_volume_mm3"})
data_metab = data_metab.rename(columns={"heart_no_cardiac_artifact_7mm_agatston": "heart_agatston"})
data_metab = data_metab.rename(columns={"heart_no_cardiac_artifact_7mm_volume_mm3": "heart_volume_mm3"})




def rename_treatment_columns(df):
    chemio_cols = [
        'Schedule_OK', 'Drug', 'Cycles',
        'Discontinued', 'Reason', 'Alternative Drug'
    ]

    immuno_cols = [
        'Drug.1', 'Cycle',
        'Discontinued.1', 'Reason.1', 'Alternative Drug.1'
    ]

    radio_cols = [
        'Prescription Dose', 'Number of Fractions',
        'EQD2 Dose',
        'Discontinued.2', 'Reason.2'
    ]

    new_cols = {}

    for col in df.columns:
        if col in chemio_cols:
            new_cols[col] = col.lower().replace(" ", "_") + "_chemo"

        elif col in immuno_cols:
            base = col.replace(".1", "")
            new_cols[col] = base.lower().replace(" ", "_") + "_immuno"

        elif col in radio_cols:
            base = col.replace(".2", "")
            new_cols[col] = base.lower().replace(" ", "_") + "_radio"

        else:
            new_cols[col] = col  # lascia invariato

    return df.rename(columns=new_cols)

data_crf = rename_treatment_columns(data_crf)
data_crf = data_crf.rename(columns={"cycle_immuno": "cycles_immuno"})



#remove empty cols
def clean_unnamed(df):
    return df.loc[:, ~df.columns.str.contains("^Unnamed")]

data_crf = clean_unnamed(data_crf)
data_dose = clean_unnamed(data_dose)
data_metab = clean_unnamed(data_metab)



# fix cycles immuno 2 columns that is eventually the tot sum
data_crf["cycles_chemo_sum"] = data_crf["cycles_chemo"]
#print(data_crf["cycles_chemo_sum"].value_counts())
data_crf.loc[data_crf["cycles_chemo_sum"] == "3+3", "cycles_chemo_sum"] = 6
data_crf.loc[data_crf["cycles_chemo_sum"] == "3+6", "cycles_chemo_sum"] = 9
data_crf.loc[data_crf["cycles_chemo_sum"] == "4 + 5", "cycles_chemo_sum"] = 9
data_crf.loc[data_crf["cycles_chemo_sum"] == "4+3", "cycles_chemo_sum"] = 7
data_crf.loc[data_crf["cycles_chemo_sum"] == "3+1", "cycles_chemo_sum"] = 4
col = data_crf.pop("cycles_chemo_sum")
data_crf.insert(data_crf.columns.get_loc("cycles_chemo") + 1, "cycles_chemo_sum", col)
#print(data_crf["cycles_chemo_sum"].value_counts())

#fix OS_new not #NUM!, #VALUE! values
data_crf.loc[data_crf["OS_new"] == "#NUM!", "OS_new"] = data_crf.loc[data_crf["OS_new"] == "#NUM!","OS"]
data_crf.loc[data_crf["OS_new"] == "#VALUE!", "OS_new"] = data_crf.loc[data_crf["OS_new"] == "#VALUE!","OS"]

print(data_crf.shape)
print(data_dose.shape)
print(data_vol.shape)
print(data_metab.shape)
print(data_crf.columns)
print(data_dose.columns)
print(data_vol.columns)
print(data_metab.columns)
None

In [ ]:
data1 = data_crf.copy()
data2 = data_dose.copy()
data3 = data_metab.copy()
data4 = data_vol.copy()
data_crf = data1.copy()
data_dose = data2.copy()
data_metab = data3.copy()
data_vol = data4.copy()

### **Datasets aligment to merge + ordering of data_crf, data_dose, data_metab**

In [ ]:
list(data_crf["ID"])
list(data_dose["ID"])
list(data_metab["ID"])
None

In [ ]:
df_match = pd.DataFrame({
    "IDCRF": data_crf["ID"],
    "IDDOSE": data_dose["ID"],
    "IDMETAB": data_metab["ID"],
    "IDVOL": data_vol["ID"]
})

df_match["ALL_EQUAL"] = (
    (df_match["IDCRF"] == df_match["IDDOSE"]) &
    (df_match["IDCRF"] == df_match["IDMETAB"]) &
    (df_match["IDCRF"] == df_match["IDVOL"])
)

print(df_match["ALL_EQUAL"].value_counts())

df_false = df_match[~df_match["ALL_EQUAL"]]
df_false.loc[:,:]
None

In [ ]:
df_match = pd.DataFrame({
    "IDCRF": data_crf["ID"],
    "IDDOSE": data_dose["ID"],
    "IDMETAB": data_metab["ID"],
    "IDVOL": data_vol["ID"]
})

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
df_match
None

In [ ]:
fix_map = {
    "LM-02001": "LM2001",
    "LM-02002": "LM2002",
    "LM-02003": "LM2003",
    "LM-02004": "LM2004",
    "LM-02005": "LM2005",
    "LM-02006": "LM2006",
    "LM-02007": "LM2007",
    "LM-02008": "LM2008",
    "LM-02009": "LM2009",
    "LM-02010": "LM2010",
    "LM-02011": "LM2011",
    "LM-02012": "LM2012",
    "LM-02013": "LM2013",
    "LM-02015": "LM2015",
    "LM-02016": "LM2016",
    "LM-02017": "LM2017",
    "LM-02018": "LM2018",
    "LM-02019": "LM2019",
    "LM-02020": "LM2020",
    "LM-02021": "LM2021",
    "LM-02022": "LM2022",
    "LM-02023": "LM2023",
    "LM-02025": "LM2025",
    "LM-02026": "LM2026",
    "LM-02027": "LM2027",
    "LM-02029": "LM2029",
    "LM-02030": "LM2030",
    "LM-02031": "LM2031",
    "LM-02032": "LM2032",
    "LM-02033": "LM2033",
    "LM-02036": "LM2036",
    "LM-02037": "LM2037",
    "LM-02038": "LM2038",
    "LM-02039": "LM2039",
    "LM-02040": "LM2040",
    "LM-02041": "LM2041",
    "LM-02042": "LM2042",
    "LM-02043": "LM2043",
    "LM-02044": "LM2044",
    "LM-02046": "LM2046",
    "LM-02047": "LM2047",
    "LM-02048": "LM2048",
    "LM-02049": "LM2049",
    "LM-02050": "LM2050",
    "LM-02051": "LM2051",
    "LM-02052": "LM2052",
    "LM-02053": "LM2053",
    "LM-02054": "LM2054",
    "LM-02055": "LM2055",
    "LM-02056": "LM2056",
    "LM-02057": "LM2057",
    "LM-02058": "LM2058",
    "LM-02059": "LM2059",
    "LM-02060": "LM2060",
    "LM-02061": "LM2061",
    "LM-02062": "LM2062",

    "LM-02060-1": "LM2060-1",
    "LM-02061-1": "LM2061-1",
    "LM-02062-1": "LM2062-1",

    "LM-02063": "LM2063",
    "LM-02064": "LM2064",
    "LM-02065": "LM2065",
    "LM-02066": "LM2066",
    "LM-02067": "LM2067",
    "LM-02068": "LM2068",
    "LM-02069": "LM2069",
    "LM-02070": "LM2070",
    "LM-02071": "LM2071",
    "LM-02072": "LM2072",
    "LM-02073": "LM2073",
    "LM-02074": "LM2074",
    "LM-02075": "LM2075",
    "LM-02076": "LM2076",
    "LM-02077": "LM2077",

    "LM-05010": "LM5010",
    "LM-05011": "LM5011",
    "LM-05012": "LM5012",
    "LM-05013": "LM5013",
    "LM-05014": "LM5014",
    "LM-05015": "LM5015",
    "LM-05018": "LM5018",
    "LM-05020": "LM5020",
    "LM-05021": "LM5021",
    "LM-05022": "LM5022",
    "LM-05023": "LM5023",
    "LM-05024": "LM5024",
    "LM-05025": "LM5025",
    "LM-05027": "LM5027",
    "LM-05028": "LM5028",
    "LM-05029": "LM5029",
    "LM-05030": "LM5030",
    "LM-05031": "LM5031",
    "LM-05032": "LM5032",
    "LM-05033": "LM5033",
    "LM-05034": "LM5034",
    "LM-05035": "LM5035",
    "LM-05037": "LM5037",
    "LM-05039": "LM5039",
    "LM-05040": "LM5040",
    "LM-05041": "LM5041",
    "LM-05042": "LM5042",
    "LM-05043": "LM5043",
    "LM-05044": "LM5044",
    "LM-05045": "LM5045",
}
df_match["IDCRF"] = df_match["IDCRF"].replace(fix_map)
df_match["IDVOL"] = df_match["IDVOL"].replace(fix_map)
df_match
fix_map2 = {
    "LM2061-2": "LM2061-1",

    "LM7003_plan1": "LM7003",
    "LM50010": "LM5010",
    "LM50011": "LM5011",
    "LM50012": "LM5012",
    "LM50013": "LM5013",
    "LM50014": "LM5014",
    "LM50015": "LM5015",
    "LM50018": "LM5018",
    "LM50020": "LM5020",
    "LM50021": "LM5021",
    "LM50022": "LM5022",
    "LM50023": "LM5023",
    "LM50024": "LM5024",
    "LM50025": "LM5025",
    "LM50027": "LM5027",
    "LM50028": "LM5028",
    "LM50029": "LM5029",
    "LM50030": "LM5030",
    "LM50031": "LM5031",
    "LM50032": "LM5032",
    "LM50033": "LM5033",
    "LM50034": "LM5034",
    "LM50035": "LM5035",
    "LM50037": "LM5037",
    "LM50039": "LM5039",
    "LM50040": "LM5040",
    "LM50041": "LM5041",
    "LM50042": "LM5042",
    "LM50043": "LM5043",
    "LM50044": "LM5044",
    "LM50045": "LM5045"
}
fix_map3 = {
    "LM2061-2": "LM2061-1"
}
df_match["IDDOSE"] = df_match["IDDOSE"].replace(fix_map2)
df_match["IDMETAB"] = df_match["IDMETAB"].replace(fix_map2)


df_match["ALL_EQUAL"] = (
    (df_match["IDCRF"] == df_match["IDDOSE"]) &
    (df_match["IDCRF"] == df_match["IDMETAB"]) &
    (df_match["IDCRF"] == df_match["IDVOL"])
)
print(df_match["ALL_EQUAL"].value_counts())
df_false = df_match[df_match["ALL_EQUAL"] == False]
df_false


In [ ]:
data_crf["ID"] = data1["ID"].replace(fix_map)
data_dose["ID"] = data2["ID"].replace(fix_map2)
data_metab["ID"] = data3["ID"].replace(fix_map2)
data_vol["ID"] = data4["ID"].replace(fix_map)
df_match = pd.DataFrame({
    "IDCRF": data_crf["ID"],
    "IDDOSE": data_dose["ID"],
    "IDMETAB": data_metab["ID"],
    "IDVOL": data_vol["ID"]
})
data_crf = data_crf.sort_values("ID").reset_index(drop=True)
data_dose = data_dose.sort_values("ID").reset_index(drop=True)
data_metab = data_metab.sort_values("ID").reset_index(drop=True)
data_vol = data_vol.sort_values("ID").reset_index(drop=True)

df_match["ALL_EQUAL"] = (
    (df_match["IDCRF"] == df_match["IDDOSE"]) &
    (df_match["IDCRF"] == df_match["IDMETAB"]) &
    (df_match["IDCRF"] == df_match["IDVOL"])
)

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
df_match
None


In [ ]:
print(data_crf.shape)
print(data_dose.shape)
print(data_metab.shape)
print(data_vol.shape)

In [ ]:
data123 = data_crf.merge(data_dose, on="ID", how="inner") \
                    .merge(data_metab, on="ID", how="inner")
print(data123.shape) # 89 = 52 +23 +13 +1
print(data123.columns)

data1234 = data_crf.merge(data_dose, on="ID", how="inner") \
                    .merge(data_metab, on="ID", how="inner")\
                    .merge(data_vol, on="ID", how="inner")
print(data1234.shape) #102  = 52 +23 +13 + 13 + 1
None

### **Continuos vs categorical variables**

In [ ]:
data_vol.columns


In [ ]:
import pandas as pd
import numpy as np

num_cols = data_crf.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = data_crf.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
force_categorical = ["KPS","schedule_ok_chemo", "cycles_chemo", "cycles_chemo_sum", "cycles_immuno", "prescription_dose_radio", "number_of_fractions_radio"] #KPS has only 5 levels
force_numeric = []
desc_cols = [
    "Date of birth",
    "inizio RT",
    "cycles_chemo",
    "alternative_drug_chemo",
    "alternative_drug_immuno",
    "data_last_FU",
    "drug_chemo",
    "drug_immuno",
    "reason_chemo",
    "reason_immuno",
    "reason_radio",
    "Note"
]
num_cols = [c for c in num_cols if c not in force_categorical + desc_cols]
cat_cols = [c for c in cat_cols if c not in force_numeric + desc_cols]
cat_cols += force_categorical
num_cols += force_numeric
num_cols = [c for c in num_cols if c not in desc_cols]
cat_cols = [c for c in cat_cols if c not in desc_cols]
desc_cols = [c for c in desc_cols if c in data_crf.columns]
print("\n Number of continuos variables in CRF data:", len(num_cols))
print("\n Number of categorical variables in CRF data:", len(cat_cols))
print("\n Number of categorical variables in CRF data:", len(desc_cols))
print("\n CONTINUOS in CRF data \n") #actually "schedule_ok_chemo" is categorical (0,1,99)
for col in sorted(num_cols):
    print(f"- {col}")
print("\n CATEGORICAL in CRF data \n")
for col in sorted(cat_cols):
    print(f"- {col}")
print("\n DESCRIPTIVE VARIABLES in CRF data \n")
for col in sorted(desc_cols):
    print(f"- {col}")

num_cols = data_dose.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = data_dose.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
print("\n Number of continuos variables in DOSE data:", len(num_cols))
print("\n Number of categorical variables in DOSE data:", len(cat_cols))
print("\n CONTINUOS in DOSE data \n")
for col in sorted(num_cols):
    print(f"- {col}")
print("\n CATEGORICAL in DOSE data \n")
for col in sorted(cat_cols):
    print(f"- {col}")

num_cols = data_metab.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = data_metab.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
print("\n Number of continuos variables in METABOLIC data:", len(num_cols))
print("\n Number of categorical variables in METABOLIC data:", len(cat_cols))
print("\n CONTINUOS in METABOLIC data \n")
for col in sorted(num_cols):
    print(f"- {col}")
print("\n CATEGORICAL in METABOLIC data \n")
for col in sorted(cat_cols):
    print(f"- {col}")



#manual summary of features types
CONT_COLS_CRF = [
    "Age",
    "Weight",
    "Height",
    "BMI",
    "SAT",
    "eqd2_dose_radio",
    "OS",
    "OS_new"
]

CAT_COLS_CRF = [
    "ID",
    "Sex",
    "Diabetes drugs",
    "Antilipidic drugs",
    "Hypertension",
    "antithrombotic",
    "Beta-Blockers",
    "Diuretics",
    "Smoking status",
    "KPS", #new
    "Stage_TNM",
    "Stage at RT",
    "Mutation",
    "Target therapy",
    "PDL-1 level",
    "N",
    "Site of Primary",
    "Mediastino",
    "CHEMO",
    "schedule_ok_chemo",
    "cycles_chemo_sum", #new
    "discontinued_chemo",
    "IMMUNO",
    "cycles_immuno",
    "discontinued_immuno",
    "prescription_dose_radio",
    "number_of_fractions_radio",
    "discontinued_radio",
    "Overall Survival at 2 years",
    "OS (yes/no)",
    "Cardiac Events Post RT",
    "pulmonary events"
]

DESCR_COLS_CRF = [
    "Date of birth",
    "Dimension",
    "drug_chemo",
    "cycles_chemo",
    "reason_chemo",
    "alternative_drug_chemo",
    "drug_immuno",
    "reason_immuno",
    "alternative_drug_immuno",
    "inizio RT",
    "reason_radio",
    "data_last_FU",
    "Note"
]

CONT_COLS_DOSE = [
    "mean_dose_aorta",
    "eud_aorta",
    "mean_dose_lad",
    "eud_lad",
    "mean_dose_pulm",
    "eud_pulm",
    "mean_dose_atriumL",
    "eud_atriumL",
    "mean_dose_atriumR",
    "eud_atriumR",
    "mean_dose_ventricleL",
    "eud_ventricleL",
    "mean_dose_ventricleR",
    "eud_ventricleR",
    "mean_dose_Heart",
    "eud_Heart",
    "mean_dose_venacava_I",
    "eud_venacavaI",
    "mean_dose_vanacava_S",
    "eud_vanacavaS",
    "mean_dose_lungs",
    "eud_lungs",
    "EDIC"
]

CAT_COLS_DOSE = [
    "ID"
]

DESCR_COLS_DOSE = []

CONT_COLS_METAB = [
    "Subcutaneous_Fat",
    "Torso_Fat",
    "Skeletal_Muscle",
    "Intermuscular_Fat",
    "heart_agatston",
    "heart_volume_mm3",
    "aorta_agatston",
    "aorta_volume_mm3",
    "LAA",
    "Volume_PTV",
    "diameter aortic arc 1 [mm]",
    "diameter aortic arc 2 [mm]",
    "diameter kidney sep [mm]",
]

CAT_COLS_METAB = [
    "ID"
]
DESCR_COLS_METAB = []

CONT_COLS_VOL = [
    "A_Aorta",
    "A_Lad",
    "A_Pulm",
    "Atrium_L",
    "Atrium_R",
    "Heart",
    "Lung_L",
    "Lung_R",
    "Lungs",
    "Ventricle_L",
    "Ventricle_R",
    "Venacava_I",
    "Venacava_S",
]


print(len(CONT_COLS_CRF), len(CAT_COLS_CRF), len(DESCR_COLS_CRF))
print(len(CONT_COLS_DOSE), len(CAT_COLS_DOSE), len(DESCR_COLS_DOSE))
print(len(CONT_COLS_METAB), len(CAT_COLS_METAB), len(DESCR_COLS_METAB))
print(len(CONT_COLS_VOL))

In [ ]:
data1 = data_crf.copy()
data2 = data_dose.copy()
data3 = data_metab.copy()
data4 = data_vol.copy()

In [ ]:
data_crf = data1.copy()
data_dose = data2.copy()
data_metab = data3.copy()
data_vol = data4.copy()

# **Step 2: distributions**

In [ ]:

import matplotlib.pyplot as plt
import seaborn as sns
import math

def plot_histograms_cont_cols(df, cont_cols, bins=100, n_cols=5, title="Continuous distributions"):

    cols = [c for c in cont_cols if c in df.columns]

    if len(cols) == 0:
        print("No continuous columns to plot")
        return

    n_rows = math.ceil(len(cols) / n_cols)

    plt.figure(figsize=(n_cols * 3.2, n_rows * 2.8))

    for i, col in enumerate(cols, 1):
        plt.subplot(n_rows, n_cols, i)

        sns.histplot(
            df[col].dropna(),
            bins=bins,
            kde=True,
            color="steelblue"
        )

        plt.title(col, fontsize=9)
        plt.xticks(rotation=45, fontsize=7)
        plt.yticks(fontsize=7)

    plt.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.show()

#plot_histograms_cont_cols(data_crf, CONT_COLS_CRF, title="CRF continuous variables")
#plot_histograms_cont_cols(data_dose, CONT_COLS_DOSE, title="DOSE continuous variables")
#plot_histograms_cont_cols(data_metab, CONT_COLS_METAB, title="METABOLIC continuous variables")
#plot_histograms_cont_cols(data_vol, CONT_COLS_VOL, title="VOL continuous variables")

#### Fix categorical cols levels

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import math

def plot_countplots_cat_cols(
    df,
    cat_cols,
    n_cols=4,
    title="Categorical distributions",
    exclude_cols=None
):

    if exclude_cols is None:
        exclude_cols = []

    cols = [
        c for c in cat_cols
        if c in df.columns and c not in exclude_cols
    ]

    if len(cols) == 0:
        print("No categorical columns to plot")
        return

    sns.set_style("whitegrid")

    n_rows = math.ceil(len(cols) / n_cols)

    fig, axes = plt.subplots(
        n_rows, n_cols,
        figsize=(n_cols * 5, n_rows * 3.8)
    )

    axes = axes.flatten()

    for i, col in enumerate(cols):

        ax = axes[i]

        counts = df[col].value_counts(dropna=False)
        total = counts.sum()

        counts = counts.sort_values(ascending=True)

        colors = sns.color_palette("Reds", len(counts))

        bars = ax.barh(
            counts.index.astype(str),
            counts.values,
            color=colors
        )

        for bar in bars:
            width = bar.get_width()
            pct = width / total * 100
            ax.text(
                width + max(counts.values)*0.01,
                bar.get_y() + bar.get_height()/2,
                f"{int(width)} ({pct:.1f}%)",
                va="center",
                fontsize=8
            )

        ax.set_title(col, fontsize=10)
        ax.set_xlabel("")
        ax.set_ylabel("")
        ax.tick_params(axis='y', labelsize=8)

    for j in range(len(cols), len(axes)):
        fig.delaxes(axes[j])

    plt.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.show()
plot_countplots_cat_cols(data_crf,CAT_COLS_CRF,title="CRF categorical variables",exclude_cols=["ID", "Stage_TNM"])


In [ ]:
data_crf.loc[data_crf["N"]=="yes","N"]="YES"
data_crf.loc[data_crf["N"]=="no","N"]="NO"
data_crf.loc[data_crf["discontinued_chemo"]=="no","discontinued_chemo"]="NO"
data_crf.loc[data_crf["IMMUNO"]=="no","IMMUNO"]="NO"
data_crf.loc[data_crf["discontinued_immuno"]=="SI","discontinued_immuno"]="YES"
data_crf.loc[data_crf["Overall Survival at 2 years"]=="yes","Overall Survival at 2 years"]="YES"
data_crf.loc[data_crf["OS (yes/no)"]=="yes","OS (yes/no)"]="YES"
data_crf.loc[data_crf["OS (yes/no)"]=="no","OS (yes/no)"]="NO"
data_crf.loc[data_crf["OS (yes/no)"]=="si","OS (yes/no)"]="YES"


#plot_countplots_cat_cols(data_crf,CAT_COLS_CRF,title="CRF categorical variables",exclude_cols=["ID", "Stage_TNM"])


In [ ]:
data1_saving_withNA =data_crf.copy()
data2_saving_withNA = data_dose.copy()
data3_saving_withNA = data_metab.copy()
data4_saving_withNA = data_vol.copy()

# **Step 3 save and download without NaNs removal**

In [ ]:
data_crf = data1_saving_withNA
data_dose = data2_saving_withNA
data_metab = data3_saving_withNA
data_vol = data4_saving_withNA

In [ ]:
print(data_crf.loc[data_crf["OS"].isna(),["ID","OS"]])
print(data_crf.loc[data_crf["OS_new"]<data_crf["OS"], ["ID","OS", "OS_new"]])
#filling OS new
print(data_crf["OS_new"].isna().sum()) #LM 5000 + LM3057, LM3060

#data_crf.loc[data_crf["OS_new"]==data_crf["OS_new"]-1, "OS_new"] = data_crf["OS"]
print(data_crf["OS_new"].isna().sum())

In [ ]:
#specific fix
data_crf.loc[data_crf["schedule_ok_chemo"] == 9,"schedule_ok_chemo"] = 1 #change in 1 #LM4012
data_crf.loc[data_crf["IMMUNO"].isna(),"IMMUNO"] = "NO" #LM1037
data_crf.loc[(data_crf["IMMUNO"]=="NO") & (data_crf["drug_immuno"].notna()),["drug_immuno"]]=np.nan #LM7010

data_crf.loc[data_crf["Stage at RT"].isna(), "Stage at RT"] = "IIIA" #LM3057

data_crf.loc[(data_crf["CHEMO"] == "NO") & (data_crf["schedule_ok_chemo"] != 99),"CHEMO"] = "YES" #LM6001
data_metab.loc[data_metab["Volume_PTV"].isna(),"Volume_PTV"]=122.07 #LM7003

In [ ]:
data_crf.loc[(data_crf["CHEMO"] == "NO") & (data_crf["cycles_chemo_sum"].isna()), "cycles_chemo_sum"] = 0
data_crf.loc[(data_crf["CHEMO"] == "NO") & (data_crf["cycles_chemo"].isna()), "cycles_chemo"] = 0
data_crf.loc[(data_crf["IMMUNO"] == "NO") & (data_crf["cycles_immuno"].isna()), "cycles_immuno"] = 0

data_crf.loc[(data_crf["CHEMO"] == "NO") & (data_crf["discontinued_chemo"].isna()), "discontinued_chemo"]="NO"
data_crf.loc[(data_crf["IMMUNO"] == "NO") & (data_crf["discontinued_immuno"].isna()), "discontinued_immuno"]="NO"

In [ ]:
#saving without removing cols
dataclean_final_withNA_fullCols = data_crf.merge(data_dose, on="ID", how="inner") \
                    .merge(data_vol, on="ID", how="inner") \
                    .merge(data_metab, on="ID", how="inner")
print(dataclean_final_withNA_fullCols.shape)
dataclean_final_withNA_fullCols.to_csv("dataclean_final_withNA_fullCols.csv", index=False) #NOTE NaNs are still present

In [ ]:
#Dimension TO BE REMOVED as the dimension already present in data_metab 'Volume_PTV'
data_crf = data_crf.drop(columns=["Weight", "Height","SAT","Stage_TNM","Mutation","Dimension","N","Site of Primary","Mediastino", "data_last_FU","Cardiac Events Post RT", "pulmonary events","Note"])

In [ ]:
dataclean_final_withNA = data_crf.merge(data_dose, on="ID", how="inner") \
                    .merge(data_vol, on="ID", how="inner") \
                    .merge(data_metab, on="ID", how="inner")
print(dataclean_final_withNA.shape)
dataclean_final_withNA.to_csv("dataclean_final_withNA.csv", index=False) #NOTE NaNs are still present
data_crf.to_csv("data_crf_final_withNA.csv", index=False)
data_dose.to_csv("data_dose_final_withNA.csv", index=False)
data_vol.to_csv("data_vol_final_withNA.csv", index=False)
data_metab.to_csv("data_metab_final_withNA.csv", index=False)

In [ ]:
dataclean_final_withNA.isna().sum()

In [ ]:
dataclean_final_withNA.loc[:,:]

In [ ]:
# August update
import numpy as np
import matplotlib.pyplot as plt
from google.colab import drive
import pandas as pd
drive.mount('/content/drive')
current_dir = '/content/drive/MyDrive/Colab Notebooks/Tesi'
%cd $current_dir
data_crf = pd.read_csv("data_crf_final_withNA.csv")
data_dose = pd.read_csv("data_dose_final_withNA.csv")
data_vol = pd.read_csv("data_vol_final_withNA.csv")
data_metab = pd.read_csv("data_metab_final_withNA.csv")

In [ ]:
data_metab.loc[data_metab["Volume_PTV"].isna(),:]


In [ ]:
data_metab.loc[data_metab["Volume_PTV"].isna(),"Volume_PTV"]=122.07 #LM7003

In [ ]:
data_metab.loc[data_metab["Volume_PTV"].isna(),:]

In [ ]:
dataclean_final_withNA = data_crf.merge(data_dose, on="ID", how="inner") \
                    .merge(data_vol, on="ID", how="inner") \
                    .merge(data_metab, on="ID", how="inner")
print(dataclean_final_withNA.shape)
dataclean_final_withNA.to_csv("dataclean_final_withNA.csv", index=False) #NOTE NaNs are still present
data_metab.to_csv("data_metab_final_withNA.csv", index=False)